In [21]:
import pandas as pd
import numpy as np
import sys
from scipy.stats import norm, t
sys.path.append("/Users/willneuner/Desktop/FINTECH545") 

In [ ]:
# how to calculate ex ante
# Decide on an ex-Ante risk model.  
# Calculate the weight of each stock in the portfolio.
# Calculate the gradient of risk wrt stocks.
# Multiply each partial derivative with that stock's weight.
# The result is the ex-ante contribution to risk.

In [ ]:
# jointly simulate from a t-distributed error system
# fit the models to get the parameters and U values for the e1 and e2 variables.  Find the U values for x
# Use the spearman correlation between the e1, e2, and X U values to fit the Gaussian copula.
# Simulate from the copula and transform back to e1, e2, and x
# use the fitted Alpha and Beta values to transform e1, e2, and X into Y1 and Y2.

In [6]:
# DISTRIBUTIONS (NORMAL AND T)
from risk_management import distributions

# mu_vector, covariance_matrix = fit_multivariate_normal_dist(x: pd.DataFrame)

# mu, sigma, nu = fit_univariate_t_dist(x)

# alpha, betas, mu, sigma, nu = t_regression(X: pd.DataFrame, y:pd.Series, add_constant:bool = True, print_summary = False)

In [7]:
# MEASUREMENTS (CORRELATION, COVARIANCE, FACTORIZATIONS, RETURNS)
from risk_management import measurements

# corr = compute_correlation(x:pd.DataFrame, method="pearson", drop_missing = False, exponentially_weighted = False, lambda_ = 0.97, ddof: int =1)

# cov = compute_covariance(x:pd.DataFrame, drop_missing = False, exponentially_weighted = False, lambda_ = 0.97, ddof:int=1)

# cov = compute_covariance_with_ew_corr(x: pd.DataFrame, corr_lambda, var_lambda)

# psd = near_psd(A: pd.DataFrame, epsilon = 0.0)
# psd = higham_psd(A: pd.DataFrame, tolerance = 1e-8, max_iterations= 100_000)

# A = cholesky_factor(x:pd.DataFrame)

# returns = compute_returns(x: pd.DataFrame, return_type = "arithmetic")


In [8]:
# RISK METRICS (VAR, ES)
from risk_management import risk_metrics

# NOTE -> all of these assume that X was a set of RETURNS, not prices

# abs_VaR, rel_VaR = univariate_normal_VaR(mean: float, std: float, alpha = 0.05)

# nu, mu, sigma = t.fit(x)
# abs_VaR, rel_VaR = univariate_t_VaR(mu: float, sigma: float, nu: float, alpha: float = 0.05)

# abs_ES, diff_ES = expected_shortfall_normal(mu:float, sigma: float, alpha = 0.05)

# mu, sigma, nu = fit_univariate_t_dist(x)
# abs_ES, diff_ES = expected_shortfall_t(mu: float, sigma: float, nu: float, alpha: float = 0.05)



In [9]:
## SIMULATIONS
from risk_management import simulations

# NOTE -> all of these assume that X was a set of RETURNS, not prices

# simulation_data = normal_monte_carlo_simulation(mean_vector, covariance_matrix, n_sims, fix_method, seed=1234) # len(cov), n_sims

# simulation_data = pca_monte_carlo_simulation(mean_vector, covariance_matrix, n_sims, explained_threshold = 0.99, seed=1234) # len(cov), n_sims

# current_prices = vector of start prices of assets, holdings = vector of how many of each asset we have
# abs_VaR, rel_VaR = monte_carlo_VaR_sim(mean_vector, covariance_matrix, current_prices, holdings, n_draws, return_type = "arithmetic", alpha = 0.05, seed = 1234) 

# sim_dataframe = VaR_ES_2_level_sim_from_copula(sample_data: pd.DataFrame, holdings: np.array, prices: np.array, fix_method, n_sims = 100_000, alpha=0.05, seed=1234)

In [10]:
from risk_management import goodness_of_fit

# r2, adjr2, Aic, Aicc, BIC

from risk_management import asset_pricing

# European, American binary, American Discontinuous div

from risk_management import portfolio_construction

# risk parity, weighted risk parity, max sharpe, efficient frontier, 

from risk_management import risk_attribution
# ex_post attribution, ex_post factor attribution


### Question 1

In [11]:
data_1 = pd.read_csv("problem1.csv")

In [136]:
returns = measurements.compute_returns(data_1, return_type="log")

In [137]:
mean, cov = distributions.fit_multivariate_normal_dist(returns)
mu, sigma, nu = distributions.fit_univariate_t_dist(returns)

print(float(mean), float(cov.iloc[0,0]))
print(mu, sigma, nu)

-0.003023851474658956 0.0005403492827998508
-0.0030238618945462436 0.02262541593783346 4361534292.275997


/var/folders/1d/dkqlxg7d3yv8ccfw3vkrtnsm0000gn/T/ipykernel_50942/2414866325.py:4: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  print(float(mean), float(cov.iloc[0,0]))


In [35]:
## better fit

# normal
norm_params = {"loc": float(mean.iloc[0]), "scale": np.sqrt(cov.iloc[0,0])}
LL_norm = goodness_of_fit.log_likelihood(norm.pdf, norm_params, returns)
norm_bic = goodness_of_fit.bic(2, LL_norm, returns.shape[0])

# t
t_params = {"loc": mu, "scale": sigma, "df": nu}
LL_t = goodness_of_fit.log_likelihood(t.pdf, t_params, returns)
t_bic = goodness_of_fit.bic(3, LL_t, returns.shape[0])

print(norm_bic, t_bic)

45.01146854040042
45.025107142279346
-84.13405912246796 -81.21689734705937


### Question 2

In [128]:
data_2 = pd.read_csv("problem2.csv")
S = data_2["Underlying"].iloc[0]
X = data_2["Strike"].iloc[0]
T = data_2["TTM"].iloc[0] / 255
rfr = data_2["RF"].iloc[0]
div = data_2["DivRate"].iloc[0]
b = rfr-div
call_price = data_2["CallPrice"].iloc[0]

In [129]:
i_vol = asset_pricing.implied_vol_gbsm(call_price, S, X, T, rfr, b, option_type="call")
i_vol

0.2999965250171499

In [130]:
put_price, put_delta, _, put_vega, *_ = asset_pricing.European_GBSM(S, X, T, i_vol, rfr, b, option_type="put")

In [131]:
_, call_delta, _, call_vega, *_ =  asset_pricing.European_GBSM(S, X, T, i_vol, rfr, b, option_type="call")

In [132]:
# long 1 call and long 1 put (straddle)

# implied vol decreases by 5%
new_ivol = i_vol -0.05

# 2 methods (recompute price or use vega)

# 1: recompute price
new_put_price, *_ = asset_pricing.European_GBSM(S, X, T, new_ivol, rfr, b, option_type="put")
new_call_price, *_ = asset_pricing.European_GBSM(S, X, T, new_ivol, rfr, b, option_type="call")

pnl = 1*(new_call_price - call_price) + 1*(new_put_price - put_price)
print(pnl)

# use vega to approximate
new_put_price = put_price + put_vega * (-0.05)
new_call_price = call_price + call_vega * (-0.05)
pnl = 1*(new_call_price - call_price) + 1*(new_put_price - put_price)
print(pnl)

-2.9543278752793967
-2.9513016838126234


### Question 3

In [125]:
# using the fitted distribution from 1 (norm was better)
# using the portfolio from 2g (1 call and 1 put)

# compute expected price TODO -> IT WAS NOT CLEAR NOT TO USE THIS (nvm -> no change in any other inputs)
S_100 = S * np.exp(100*mean)

# compute call and put prices
X = data_2["Strike"].iloc[0]
T = (data_2["TTM"].iloc[0]-100) / 255
rfr = data_2["RF"].iloc[0]
div = data_2["DivRate"].iloc[0]
b = rfr-div
i_vol = 0.30

new_put_price, *_ = asset_pricing.European_GBSM(S, X, T, i_vol, rfr, b, option_type="put")
new_call_price, *_ = asset_pricing.European_GBSM(S, X, T, i_vol, rfr, b, option_type="call")

pnl = 1*(new_call_price - call_price) + 1*(new_put_price - put_price)
print(pnl)

-7.306503195399678


In [134]:
### NOTE-> delta normal VaR and ES assume linear payoff, but options don't do that. As such, we are liable to run into issues

## VaR and ES
asset_prices = np.array([call_price, put_price])
holdings = np.array([1, 1])
portfolio_value = holdings @ asset_prices
weights = asset_prices * holdings / portfolio_value
underlying_prices = np.array([S])
underlying_indices = np.array([0,0])

deltas = np.array([call_delta, put_delta])

VaR = risk_metrics.delta_normal_var(asset_prices, weights, underlying_prices, underlying_indices, deltas, cov, alpha=0.05)
ES = risk_metrics.delta_normal_es(asset_prices, weights, underlying_prices, underlying_indices, deltas, cov, alpha=0.05)
VaR, ES
# VaR*np.sqrt(100), ES*np.sqrt(100)

# we should absolutely be taking this position. The 100 day adjusted Expected shortfall is $2.37 while the expected return is $9.41 
# our initial portfolio value is $17.78, so we expect to make a 53% return on our portfolio, and our expected shortfall (which doesn't account for mean returns)
# shows a 13.3% loss. Overall, we expect a large gain even in the worst case scenario

(np.float64(0.18951366954380683), np.float64(0.23765778727086276))

In [146]:
sims = simulations.normal_monte_carlo_simulation(mean, cov, n_sims=100_000, fix_method=measurements.higham_psd)
T = data_2["TTM"].iloc[0] / 255

asset_prices = np.array([call_price, put_price])
holdings = np.array([1, 1])
portfolio_value = holdings @ asset_prices

pnls = []
for log_return in sims:
    S_new = S * np.exp(log_return)
    
    new_put_price, *_ = asset_pricing.European_GBSM(S_new, X, T-1/255, i_vol, rfr, b, option_type="put")
    new_call_price, *_ = asset_pricing.European_GBSM(S_new, X, T-1/255, i_vol, rfr, b, option_type="call")

    pnl = 1*new_call_price + 1 * new_put_price - portfolio_value
    pnls.append(pnl)

pnls = np.array(pnls)
VaR = -np.percentile(pnls, 0.05*100)
ES = -np.mean(pnls[pnls <= np.percentile(pnls, 0.05*100)])

### Question 4

In [153]:
data_4 = pd.read_csv("problem4.csv")
weights = np.array([0.3, 0.45, 0.25])

In [154]:
cov = measurements.compute_covariance(data_4, exponentially_weighted=True, lambda_=0.94)
cov

,A,B,C
A,0.000103,0.000113,-0.000064
B,0.000113,0.000379,0.000116
C,-0.000064,0.000116,0.003240


In [173]:
vol = np.sqrt(weights.T @ cov @ weights)
csd = weights * (cov.dot(weights)) / vol
print(csd)
print(csd / vol)

A    0.001074
B    0.005732
C    0.011511
dtype: float64
A    0.058623
B    0.312944
C    0.628433
dtype: float64


In [189]:
means = np.mean(data_4, axis=0)
rfr = 0.0525
days = 365
# daily_rfr = rfr / days
# daily_rfr = (1+rfr)**(1/365)-1
daily_rfr = np.log(1+rfr)/365

In [190]:
portfolio_daily_return = weights.T @ means
(portfolio_daily_return - daily_rfr)/vol

np.float64(0.19964613449616436)

In [195]:
# max sharpe ratio portfolio
portfolio_construction.compute_max_sharpe_weights(means, cov, daily_rfr) #TODO -> ensure everything is on the same time scale

Optimization terminated successfully    (Exit mode 0)
            Current function value: -0.29328865775037577
            Iterations: 8
            Function evaluations: 32
            Gradient evaluations: 8


array([0.        , 0.96480634, 0.03519366])

### Question 5

In [200]:
data_5 = pd.read_csv("problem5.csv")
holdings = np.array([1,1,1,1])

In [199]:
returns = measurements.compute_returns(data_5, return_type="arithmetic")

In [214]:
means = []
stds = []
dfs = []
ESs = []
for column in returns:
    mu, sigma, nu = distributions.fit_univariate_t_dist(returns.loc[:, column])
    means.append(mu)
    stds.append(sigma)
    dfs.append(nu)
    ES, _ = risk_metrics.expected_shortfall_normal(mu, sigma)
    ESs.append(ES)

ESs

[np.float64(0.021251891780204982),
 np.float64(0.03814130489788621),
 np.float64(0.0947559791888307),
 np.float64(0.05000200196291665)]

array([107.66834882,  46.42810259, 102.9212364 ,  78.3862182 ])

In [213]:
simulations.VaR_ES_2_level_sim_from_copula(returns, holdings, data_5.iloc[-1].to_numpy(), fix_method=measurements.higham_psd, n_sims=100_000)

,Stock,VaR95,ES95,VaR95_Pct,ES95_Pct
0,A,1.888941,2.381402,0.017544,0.022118
1,B,1.667394,2.106587,0.035913,0.045373
2,C,9.184298,11.455378,0.089236,0.111302
3,D,3.534012,4.447628,0.045085,0.056740
4,Total,10.129821,12.678380,0.030202,0.037800


In [245]:
def compute_PIT(returns: pd.DataFrame, models: dict, marginal="normal"):
    """
        Convert returns into Uniform [0,1] for later use in converting to standard normal for simulation
    """
    U = pd.DataFrame(index=returns.index)
    if marginal == "normal":
        for col in returns.columns:
            params = models[col]
            U[col] = norm.cdf(returns[col], **params)
    elif marginal == "t":
        for col in returns.columns:
            params = models[col]
            U[col] = t.cdf(returns[col], **params)
    else:
        raise ValueError("marginal must be 'normal' or 't'")
    return U

def gaussianize(U: pd.DataFrame) -> pd.DataFrame: # get the z score from the cdf value U's
    """
        Convert U [0,1] to normal distribution Z scores
    """
    Z = pd.DataFrame(norm.ppf(U, loc=0, scale=1), columns=U.columns, index=U.index)
    return Z

def invert_marginals(Z: np.ndarray, models: dict, columns: list, marginal="normal") -> pd.DataFrame:
    """
        Convert Z scores back into simulated returns from original fit distributions
    """
    simulated_U = norm.cdf(Z)
    simulated_returns = pd.DataFrame(index=np.arange(Z.shape[0]), columns=columns)
    
    if marginal == "normal":
        for i, col in enumerate(columns):
            params = models[col]
            simulated_returns[col] = norm.ppf(simulated_U[:, i], **params)
    elif marginal == "t":
        for i, col in enumerate(columns):
            params = models[col]
            simulated_returns[col] = t.ppf(simulated_U[:, i], **params)
    return simulated_returns


def compute_portfolio_returns(simulated_returns: pd.DataFrame,
                              prices: np.ndarray,
                              holdings: np.ndarray) -> pd.Series:
    simulated_values = simulated_returns * prices + prices # assumes arithmetic returns
    total_values = simulated_values.dot(holdings)
    initial_value = prices.dot(holdings)
    portfolio_returns = (total_values - initial_value) / initial_value
    return portfolio_returns

def compute_var_es(return_series: pd.Series, alpha=0.05):
    sorted_returns = np.sort(return_series)
    n = len(sorted_returns)
    idx = int(np.floor(alpha * n))
    VaR_pct = -sorted_returns[idx]
    ES_pct = -sorted_returns[:idx+1].mean()
    return VaR_pct, ES_pct




In [277]:
def copula_var_es(returns: pd.DataFrame,
                  prices: np.ndarray,
                  holdings: np.ndarray,
                  fix_method,
                  n_sims=100_000,
                  alpha=0.05,
                  seed=1234,
                  marginal="normal",
                  ):
    """
    Gaussian copula VaR/ES with optional t-distribution marginals.
    marginal: "normal" or "t"
    """

    models = {}
    if marginal == "normal":
        for col in returns.columns:
            mu = returns[col].mean()
            sigma2 = returns[col].var()
            models[col] = {"loc":mu, "scale": np.sqrt(sigma2)}
    elif marginal == "t":
        for col in returns.columns:
            mu, sigma, nu = distributions.fit_univariate_t_dist(returns[col].values)
            models[col] = {"loc":mu, "scale": sigma, "df": nu}
    else:
        raise ValueError("marginal must be 'normal' or 't'")

    # PIT
    U = compute_PIT(returns, models, marginal)

    # Gaussianize
    Z_data = gaussianize(U)

    # Correlation
    corr = measurements.compute_correlation(Z_data, method="spearman")

    # Simulate
    # Z_sim = simulate_gaussian_copula(corr, n_sims, seed)
    simulated_Zs = simulations.normal_monte_carlo_simulation(mean_vector=np.zeros(len(models)), covariance_matrix=corr, n_sims=n_sims, fix_method=fix_method, seed=seed).T

    # Invert marginals
    sim_returns = invert_marginals(simulated_Zs, models, returns.columns.tolist(), marginal)

    # Portfolio returns
    portfolio_returns = compute_portfolio_returns(sim_returns, prices, holdings)

    # VaR / ES
    VaR_pct, ES_pct = compute_var_es(portfolio_returns, alpha)
    portfolio_value = prices.dot(holdings)
    VaR = VaR_pct * portfolio_value
    ES = ES_pct * portfolio_value

    return VaR, ES, VaR_pct, ES_pct


In [278]:
type(returns.iloc[:, 0:1])

pandas.core.frame.DataFrame

In [279]:
# ES for 1, 2, 3, 4
for i in range(4):
    VaR, ES, VaR_pct, ES_pct = copula_var_es(returns.iloc[:, i:i+1], data_5.iloc[-1].to_numpy()[i:i+1], holdings[i:i+1], fix_method=measurements.higham_psd, marginal="t")
    print(f"ES: {i}", ES)


ES: 0 2.423942477672393
ES: 1 2.186419763977739
ES: 2 12.739040715464078
ES: 3 4.572106500421381


In [280]:
# ES for 1+2
VaR, ES, VaR_pct, ES_pct = copula_var_es(returns.iloc[:, :2], data_5.iloc[-1].to_numpy()[:2], holdings[:2], fix_method=measurements.higham_psd, marginal="t")
ES

np.float64(3.901934094660188)

In [282]:
# ES for 3, 4
VaR, ES, VaR_pct, ES_pct = copula_var_es(returns.iloc[:, 2:], data_5.iloc[-1].to_numpy()[2:], holdings[2:], fix_method=measurements.higham_psd, marginal="t")
ES

np.float64(12.895502078867331)

In [283]:
# ES for total portfolio
VaR, ES, VaR_pct, ES_pct = copula_var_es(returns, data_5.iloc[-1].to_numpy(), holdings, fix_method=measurements.higham_psd, marginal="t")
ES

np.float64(13.549427608137293)

### Question 6

In [353]:
S = 100
X = 100
rfr = 0.0525
div=0
mean_return = 0.1
vol = 0.25
i_vol = 0.25
T = 1
b = rfr-div
# hold portfolio to expiration
# can short, but no weights < -1

# portfolio = stock and stock puts
put_price, *_ = asset_pricing.European_GBSM(S, X, T, i_vol, rfr, b, option_type="put")
S, put_price

(100, np.float64(7.347758135590375))

In [354]:
returns = simulations.normal_monte_carlo_simulation(np.array([mean_return]), np.array([[vol**2]]), n_sims=100_000, fix_method=measurements.higham_psd).T

# new_S = S * np.exp(returns[:, 0]) # NOTE -> don't use log returns for long term
new_S = (1+returns[:, 0]) * S

new_put_values = np.maximum(0, X - new_S)

put_returns = (new_put_values - put_price) / put_price

In [359]:
returns.mean()

np.float64(0.10024642724903447)

In [360]:
put_returns

array([-1.        ,  1.69121438, -1.        , ..., -1.        ,
        1.79177257, -1.        ], shape=(100000,))

In [361]:
total_returns = np.concat([returns, put_returns[:, np.newaxis]], axis=1)
cov = measurements.compute_covariance(pd.DataFrame(total_returns))

In [362]:
means, cov

(array([[ 0.10024643],
        [-0.31153457]]),
           0         1
 0  0.062583 -0.292962
 1 -0.292962  2.302335)

In [363]:
means = total_returns.mean(axis=0)[:, np.newaxis]

In [365]:
# compute max sharpe
w = portfolio_construction.compute_max_sharpe_weights(means, cov, rfr, weight_bounds=(-1, None))
w

Optimization terminated successfully    (Exit mode 0)
            Current function value: -0.19643355129642806
            Iterations: 15
            Function evaluations: 50
            Gradient evaluations: 15


array([ 1.09841362, -0.09841362])

In [366]:
(w @ means - rfr) / np.sqrt(w.T @ cov @ w)

array([0.19643355])

In [368]:
def expected_shortfall(w, means, cov, alpha = 0.05):
    mu = w @ means
    sigma = np.sqrt(w.T @ cov @ w)
    abs_ES, _ = risk_metrics.expected_shortfall_normal(mu, sigma, alpha)
    return abs_ES

optimal_es_portfolio = portfolio_construction.efficient_frontier(means, cov, pos_weights=False, custom_min_func=lambda a, b, c: expected_shortfall(a,b,c, alpha=0.01))
print(optimal_es_portfolio)
print(expected_shortfall(optimal_es_portfolio, means, cov))

[0.88517145 0.11482855]
[0.22657812]
